In [0]:
from pyspark.sql.functions import col, sum, count, avg, round, current_timestamp

# Connection
spark.conf.set(
    "fs.azure.account.key.stretailadls01.dfs.core.windows.net",
    "account-key"
)

# Paths
silver_path = "abfss://retail-project@stretailadls01.dfs.core.windows.net/silver"
gold_path   = "abfss://retail-project@stretailadls01.dfs.core.windows.net/gold"

# Silver data read చేయి
df_sales     = spark.read.format("delta").load("abfss://retail-project@stretailadls01.dfs.core.windows.net/silver/sales/")
df_stores    = spark.read.format("delta").load("abfss://retail-project@stretailadls01.dfs.core.windows.net/silver/dimensions/stores/")
df_products  = spark.read.format("delta").load("abfss://retail-project@stretailadls01.dfs.core.windows.net/silver/dimensions/products/")
df_customers = spark.read.format("delta").load("abfss://retail-project@stretailadls01.dfs.core.windows.net/silver/dimensions/customers/")

print("Silver Sales rows:", df_sales.count())

# Sales + Stores join చేయి
df_joined = df_sales.join(df_stores, on="store_id", how="left")

# ========== KPI 1 — Daily Revenue per Store ==========
df_daily_revenue = df_joined.groupBy("store_id", "store_name", "city", "region", "sale_date")\
    .agg(
        round(sum("total_amount"), 2).alias("total_revenue"),
        count("sale_id").alias("total_orders"),
        round(avg("total_amount"), 2).alias("avg_order_value")
    )\
    .withColumn("ingested_at", current_timestamp())

df_daily_revenue.show()

df_daily_revenue.write.format("delta").mode("overwrite")\
    .save("abfss://retail-project@stretailadls01.dfs.core.windows.net/gold/daily_revenue/")

print("✅ KPI 1 — Daily Revenue Done!")

# ========== KPI 2 — Product Performance ==========
df_product_perf = df_sales.groupBy("product_name", "category")\
    .agg(
        sum("quantity").alias("total_quantity_sold"),
        round(sum("total_amount"), 2).alias("total_revenue"),
        count("sale_id").alias("total_orders")
    )\
    .withColumn("ingested_at", current_timestamp())

df_product_perf.show()

df_product_perf.write.format("delta").mode("overwrite")\
    .save("abfss://retail-project@stretailadls01.dfs.core.windows.net/gold/product_performance/")

print("✅ KPI 2 — Product Performance Done!")

# ========== KPI 3 — Store Performance ==========
df_store_perf = df_joined.groupBy("store_id", "store_name", "city", "region", "state")\
    .agg(
        round(sum("total_amount"), 2).alias("total_revenue"),
        count("sale_id").alias("total_orders"),
        round(avg("total_amount"), 2).alias("avg_order_value")
    )\
    .withColumn("ingested_at", current_timestamp())

df_store_perf.show()

df_store_perf.write.format("delta").mode("overwrite")\
    .save("abfss://retail-project@stretailadls01.dfs.core.windows.net/gold/store_performance/")

print("✅ KPI 3 — Store Performance Done!")
print("🎉 Silver to Gold — Complete!")

Silver Sales rows: 15
+--------+-----------------+---------+------+----------+-------------+------------+---------------+--------------------+
|store_id|       store_name|     city|region| sale_date|total_revenue|total_orders|avg_order_value|         ingested_at|
+--------+-----------------+---------+------+----------+-------------+------------+---------------+--------------------+
|    S003|  BigMart Chennai|  Chennai| South|2024-01-01|      1000.00|           1|        1000.00|2026-04-20 04:18:...|
|    S003|  BigMart Chennai|  Chennai| South|2024-01-03|      1620.00|           2|         810.00|2026-04-20 04:18:...|
|    S002|BigMart Bangalore|Bangalore| South|2024-01-02|      2220.00|           2|        1110.00|2026-04-20 04:18:...|
|    S002|BigMart Bangalore|Bangalore| South|2024-01-03|      1000.00|           1|        1000.00|2026-04-20 04:18:...|
|    S001|BigMart Hyderabad|Hyderabad| South|2024-01-03|       750.00|           1|         750.00|2026-04-20 04:18:...|
|    S001|

In [0]:
print("=== Gold Folders ===")
dbutils.fs.ls("abfss://retail-project@stretailadls01.dfs.core.windows.net/gold/store_performance/")



=== Gold Folders ===


[FileInfo(path='abfss://retail-project@stretailadls01.dfs.core.windows.net/gold/store_performance/_delta_log/', name='_delta_log/', size=0, modificationTime=1776658737000),
 FileInfo(path='abfss://retail-project@stretailadls01.dfs.core.windows.net/gold/store_performance/part-00000-f4aee9b8-60c1-49e0-af1c-4f10e7ca5bb1.c000.snappy.parquet', name='part-00000-f4aee9b8-60c1-49e0-af1c-4f10e7ca5bb1.c000.snappy.parquet', size=2639, modificationTime=1776658738000)]